# 추천 에이전트 (2) — 아이템 DB + 다중 도구

실제 추천 시스템은 여러 검색/랭킹 전략을 조합한다. 이 노트북은 네 가지 도구를 가진 에이전트를 만든다:
1. **유사 아이템 검색** — 참조 아이템의 카테고리/가격으로 비슷한 것 찾기
2. **SQL 검색** — 자연어를 SQL 로 바꿔 DB 조회 (Text-to-SQL 에이전트를 도구로)
3. **랭킹** — 인기도 + 사용자 취향 점수로 재정렬
4. **메모리** — 사용자 정보/취향 조회·저장

> 실제 서비스는 대용량 아이템 데이터셋을 쓰지만, 여기서는 **작은 샘플 데이터**를 노트북에서 생성해 바로 실행 가능하게 한다 (self-contained).

> `OPENAI_API_KEY` 필요.

## 환경 변수 준비

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY 가 .env 에 없습니다"

## 0. 샘플 아이템 DB 생성

영화/TV 아이템을 `reviews` 테이블로 SQLite 에 저장한다. 실전 리뷰 데이터셋에서 흔히 쓰는 스키마 형태:
`title, main_category, categories(JSON), average_rating, rating_number, price`.

In [ ]:
import pandas as pd
import sqlite3
import json

rows = [
    ("Glee", "Prime Video", ["Drama", "Comedy", "Musical"], 4.5, 1200, "$0.00"),
    ("Breaking Bad", "Prime Video", ["Drama", "Crime", "Thriller"], 4.9, 5400, "$1.99"),
    ("The Office", "Prime Video", ["Comedy"], 4.8, 3100, "$0.00"),
    ("Friends", "Prime Video", ["Comedy", "Romance"], 4.7, 4200, "$2.99"),
    ("Sherlock", "Prime Video", ["Drama", "Crime", "Mystery"], 4.8, 2600, "$1.99"),
    ("La La Land", "Movies", ["Drama", "Musical", "Romance"], 4.4, 2200, "$3.99"),
    ("Inception", "Movies", ["Action", "Sci-Fi", "Thriller"], 4.8, 6100, "$3.99"),
    ("The Godfather", "Movies", ["Drama", "Crime"], 4.9, 7000, "$4.99"),
    ("Parasite", "Movies", ["Drama", "Thriller"], 4.7, 3300, "$3.99"),
    ("Planet Earth", "Prime Video", ["Documentary"], 4.9, 1800, "$0.00"),
]
df = pd.DataFrame(rows, columns=[
    "title", "main_category", "categories", "average_rating", "rating_number", "price"
])
df["categories"] = df["categories"].apply(json.dumps)  # 리스트 → JSON 문자열

conn = sqlite3.connect("movie_reviews.db")
df.to_sql("reviews", conn, if_exists="replace", index=False)
conn.close()
print(f"movie_reviews.db 생성 ({len(df)} rows)")

## 1. 유사 아이템 검색 도구

참조 아이템의 `main_category` 와 가격대(±20%)로 비슷한 아이템을 찾아, 카테고리 겹침 수와 평점으로 정렬한다.
[basics 복습] `@tool` 로 도구화 — SQL 쿼리를 동적으로 조립한다.

In [ ]:
import json
import sqlite3
import pandas as pd
from langchain_core.tools import tool

DB_PATH = "movie_reviews.db"

@tool
def find_similar_items(reference_item: str, price_min: float = None,
                       price_max: float = None, limit: int = 5):
    """Find items similar to a reference item by category and price range.

    Args:
        reference_item: title of the reference item
        price_min/price_max: price filters (default: reference price ±20%)
        limit: max results
    """
    conn = sqlite3.connect(DB_PATH)
    ref = pd.read_sql(
        "SELECT main_category, categories, price FROM reviews WHERE title = ? LIMIT 1",
        conn, params=[reference_item],
    )
    if ref.empty:
        conn.close()
        return f"Reference item '{reference_item}' not found."

    main_category = ref.iloc[0]["main_category"]
    try:
        categories = json.loads(ref.iloc[0]["categories"])
    except Exception:
        categories = []
    price = float(str(ref.iloc[0]["price"]).replace("$", "").replace(",", "") or 0)
    if price_min is None:
        price_min = price * 0.8
    if price_max is None:
        price_max = price * 1.2

    cand = pd.read_sql(
        "SELECT DISTINCT title, main_category, categories, average_rating, price "
        "FROM reviews WHERE title != ? AND main_category = ? "
        "AND CAST(REPLACE(price,'$','') AS REAL) BETWEEN ? AND ? "
        "ORDER BY average_rating DESC LIMIT ?",
        conn, params=[reference_item, main_category, price_min, price_max, limit * 5],
    )
    conn.close()

    # 카테고리 겹침 수로 재정렬
    def overlap(cat_json):
        try:
            return len(set(json.loads(cat_json)) & set(categories))
        except Exception:
            return 0
    cand["category_overlap"] = cand["categories"].apply(overlap)
    result = cand.sort_values(["category_overlap", "average_rating"], ascending=[False, False]).head(limit)
    return result.to_dict(orient="records")

In [ ]:
# 동작 확인
find_similar_items.invoke({"reference_item": "Glee", "limit": 3})

## 2. SQL 검색 도구

[basics 복습] SQL RAG 에서 배운 것처럼, 자연어 질문을 SQL 로 바꿔 실행하는 에이전트를 만들고 그것을 다시 **도구로** 감싼다 (`SQLDatabaseToolkit` + `create_react_agent`).

In [ ]:
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent

llm = ChatOpenAI(model="gpt-4o")
db = SQLDatabase.from_uri(f"sqlite:///{DB_PATH}")
sql_tools = SQLDatabaseToolkit(db=db, llm=llm).get_tools()

sql_executor = create_react_agent(
    llm, sql_tools,
    prompt="You are an expert SQL assistant. Understand the request, check schema if needed, "
           "generate and run a query, and present results clearly. Handle Korean/English. "
           "Always include 'title IS NOT NULL' unless told otherwise.",
)

from langchain_core.tools import tool

@tool
def sql_search_tool(query: str) -> str:
    """Search the item DB using natural language (converted to SQL)."""
    try:
        response = sql_executor.invoke({"messages": [("user", query)]})
        return response["messages"][-1].content
    except Exception as e:
        return f"Error: {e}"

## 3. 랭킹 도구

후보 아이템을 **인기도(평점·리뷰수)** 와 **사용자 취향 점수(카테고리 선호도)** 로 재정렬한다. 취향은 저장소(store)에서 가져온다.

In [ ]:
from langgraph.store.memory import InMemoryStore
from typing import List, Dict
from pydantic import BaseModel
from langchain_core.runnables import RunnableConfig

store = InMemoryStore()
store.put(
    ("users",), "user_123",
    {
        "preferred_categories": {"Drama": 0.9, "Comedy": 0.8, "Documentary": 0.6,
                                  "Crime": 0.5, "Sci-Fi": 0.3},
        "memory": ["이 사용자가 최근에 본 영화는 'Glee'입니다.", "주로 드라마와 코미디를 선호합니다."],
    },
)

class RankingInput(BaseModel):
    titles: List[str]

@tool(args_schema=RankingInput)
def ranking_tool(titles: List[str], config: RunnableConfig) -> Dict:
    """Rank the given item titles by popularity and the user's category preferences."""
    conn = sqlite3.connect(DB_PATH)
    placeholder = ", ".join(f'"{t}"' for t in titles)
    items = pd.read_sql(
        f"SELECT title, categories, AVG(average_rating) AS avg_rating, "
        f"SUM(rating_number) AS total_count FROM reviews "
        f"WHERE title IN ({placeholder}) GROUP BY title",
        conn,
    )
    conn.close()

    prefs = {}
    user = store.get(("users",), config["configurable"].get("user_id"))
    if user:
        prefs = user.value
    cat_prefs = prefs.get("preferred_categories", {})

    def pref_score(cat_json):
        try:
            cats = json.loads(cat_json)
        except Exception:
            return 0.5
        scores = [cat_prefs.get(c, 0.5) for c in cats] or [0.5]
        return (max(scores) + sum(scores) / len(scores)) / 2

    ranked = []
    for _, row in items.iterrows():
        avg = row["avg_rating"] if pd.notna(row["avg_rating"]) else 3.0
        cnt = row["total_count"] if pd.notna(row["total_count"]) else 1
        popularity = avg * (cnt ** 0.1)
        pscore = pref_score(row["categories"])
        ranked.append({
            "title": row["title"], "avg_rating": float(avg),
            "preference_score": round(pscore, 3),
            "final_score": round(popularity * (1 + 0.3 * pscore), 3),
        })
    ranked.sort(key=lambda x: x["final_score"], reverse=True)
    return {"ranked_items": ranked, "user_memory": prefs.get("memory", [])}

In [ ]:
# 동작 확인
ranking_tool.invoke(
    {"titles": ["Glee", "Breaking Bad", "The Office"]},
    config={"configurable": {"user_id": "user_123"}},
)

## 4. 메모리 도구
[basics 복습] 저장소에서 사용자 정보 조회 + 새 메모리(중요 발화) 저장.

In [ ]:
from langgraph.config import get_store

def get_user_info(config: RunnableConfig) -> str:
    """Look up user info (memory & preferences) from the store."""
    store = get_store()
    user_info = store.get(("users",), config["configurable"].get("user_id"))
    return str(user_info.value) if user_info else "Unknown user"

def save_memory(memory_items: list[str], config: RunnableConfig = None) -> str:
    """Save memory sentences (facts/preferences from conversation) to the user's store."""
    store = get_store()
    user_id = config["configurable"].get("user_id")
    user_info = store.get(("users",), user_id)
    user_data = user_info.value if user_info else {}
    user_data.setdefault("memory", [])
    user_data["memory"] = list(set(user_data["memory"] + memory_items))
    store.put(("users",), user_id, user_data)
    return f"Added {len(memory_items)} new memories."

## 🚩 추천 에이전트 조립

네 도구를 모두 가진 ReAct 에이전트. 프롬프트로 **의도 파악 → 아이템 수집 → 랭킹** 흐름을 지시한다.
[basics 복습] `create_react_agent(tools=[...], store=, checkpointer=)`.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

recommender_tools = [find_similar_items, sql_search_tool, ranking_tool, get_user_info, save_memory]

RECOMMENDER_PROMPT = """You are an intelligent recommendation assistant (respond in Korean).

Decision process:
1. Understand intent:
   - Specific item mentioned -> find_similar_items
   - General category/criteria -> sql_search_tool
   - Follow-up on previous items -> reuse them
2. Gather candidate items with the search tool(s).
3. ALWAYS rank with ranking_tool, then present with reasoning.

Memory: if the user shares preferences/name/habits, call save_memory with natural Korean sentences.
Explain your tool choices and ranking criteria briefly.
"""

recommender = create_react_agent(
    model=llm, tools=recommender_tools, store=store,
    prompt=RECOMMENDER_PROMPT, checkpointer=InMemorySaver(),
)

In [ ]:
from IPython.display import Image, display

try:
    display(Image(recommender.get_graph().draw_mermaid_png()))
except Exception:
    print(recommender.get_graph().draw_mermaid())

## 테스트
"Glee 비슷한 거 추천해줘" → 유사검색 → 랭킹 → 추천.

In [ ]:
config = {"configurable": {"user_id": "user_123", "thread_id": "1"}}
response = recommender.invoke(
    {"messages": [{"role": "user", "content": "Glee 랑 비슷한 영화 추천해줘"}]},
    config,
)
response["messages"][-1].pretty_print()

## 정리

- 실전 추천 = **여러 검색/랭킹 전략을 도구로** 조합한 ReAct 에이전트
- 유사검색(카테고리·가격) / SQL검색(자연어→SQL) / 랭킹(인기+취향) / 메모리(취향 저장·조회)
- 사용자 취향은 `InMemoryStore` 에 두고 랭킹·추천에 반영
- 프롬프트로 도구 선택 흐름(의도→수집→랭킹)을 지시

이로써 메모리(1) + 아이템 DB·다중도구(2)로 추천 에이전트의 핵심을 다뤘다.
다음: 지식 그래프를 활용하는 **GraphRAG**.